# Twisted Truths Factory — GPU Pipeline

In [ ]:
import subprocess
subprocess.run(['pip','install','-q','chatterbox-tts',
    'diffusers','transformers','accelerate',
    'soundfile','torch','tqdm'], check=False)
print('✅ Installed')

In [ ]:
import os, json, torch, soundfile as sf
from pathlib import Path
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

BASE = Path('/kaggle/working')
for ch in ['twistedtruths','crimeledger','mindtactics']:
    for d in ['audio','images','video']:
        (BASE/ch/d).mkdir(parents=True, exist_ok=True)
print('✅ Dirs ready')

In [ ]:
SCRIPTS = {
  "twistedtruths": {
    "title": "She Trusted Her Best Friend With Her Business \u2014 Then She Stole Everything",
    "narration_wpm": 155,
    "voice": "female_warm",
    "scenes": [
      {
        "id": "s01_hook",
        "duration": 35,
        "narration": "She handed the keys of her multi-million dollar business to her best friend, the person who had stood by her through everything. She thought she was getting a lifeline. She was actually signing a death warrant for her career. In the world of high-stakes partnerships, the person holding the ladder is often the one preparing to pull it down. This is the story of how a single signature, born of grief and trust, turned a lifelong friendship into a calculated corporate execution.",
        "image_prompt": "Cinematic wide shot of a woman looking out a rain-streaked office window at a dark city skyline, silhouette, low-key lighting, moody blue and orange tones.",
        "pexels_query": "woman silhouette office window rain city",
        "music_mood": "tense mystery"
      },
      {
        "id": "s02_foundation",
        "duration": 75,
        "narration": "Claire spent five years building Sienna Botanicals from a tiny kitchen experiment into a boutique organic skincare brand valued at two million dollars. It was her life's work. By late 2025, the rapid scaling was suffocating her. Enter Danielle. Danielle was Claire's maid of honor, her confidante, and a seasoned operational manager. When Claire's father fell terminally ill, Danielle stepped in, offering to manage daily operations so Claire could spend those final precious months at her father's bedside. It felt like a miracle. Danielle proposed a fifty-fifty partnership to 'align their interests.' Claire, blinded by grief and gratitude, signed without a second thought. She trusted Danielle with her life; therefore, she trusted her with her business.",
        "image_prompt": "Two women business partners laughing over a notebook in a modern sunlit startup office, organic textures, warm golden hour lighting, cinematic bokeh.",
        "pexels_query": "two women business partners laughing office",
        "music_mood": "warm hopeful"
      },
      {
        "id": "s03_first_cracks",
        "duration": 90,
        "narration": "The transition was seamless at first. But three months after Danielle took the reins, Claire returned to the office part-time. The atmosphere had shifted. The team Claire had hand-picked looked down when she walked by. Danielle had moved Claire's personal belongings out of the main corner office into a smaller, windowless space down the hall, calling it a 'quiet environment for her recovery.' Claire brushed it off as overprotectiveness. But then came the financial discrepancy. While reviewing quarterly accounts, Claire noticed a series of large wire transfers to an unfamiliar entity called Vanguard Consulting Group. The total was ninety-five thousand dollars. All authorized by Danielle. When Claire asked about it, Danielle smiled and said it was a 'tax mitigation strategy.' It was classic gaslighting. Claire wanted to believe her best friend. But that night, Claire's intuition kept her awake.",
        "image_prompt": "A woman looking at a laptop screen in a dark room, blue light reflecting on her face, worried expression, close-up, high contrast.",
        "pexels_query": "woman looking at laptop dark room worried",
        "music_mood": "unsettling drone"
      },
      {
        "id": "s04_the_trap",
        "duration": 105,
        "narration": "The wire transfers weren't tax strategies. They were systematic drains on the company's operating capital. But when Claire tried to log back into the banking portal the next morning to print the evidence, her access code failed. Locked. She called the bank, only to be told that her name had been removed as an authorized administrator. The authorization had been changed via an amendment to the operating agreement\u2014an amendment that Claire had signed months earlier, tucked inside a thick stack of 'routine operational paperwork' she'd signed while grieving. Danielle hadn't just taken the money; she had legally stripped Claire of her financial oversight. Therefore, Claire was now a guest in her own company, with no power to stop the bleeding.",
        "image_prompt": "Macro shot of a pen signing a legal document, focus on the ink spreading on paper, dramatic shadows, shallow depth of field.",
        "pexels_query": "pen signing document close up",
        "music_mood": "dark suspense"
      },
      {
        "id": "s05_the_staff_betrayal",
        "duration": 90,
        "narration": "Danielle hadn't just taken the bank accounts. She had taken the hearts and minds of the staff. Over the previous six months, Danielle had quietly painted Claire as an unstable, absentee founder who was draining the company's resources for personal matters. She had implemented a new bonus structure that made the design team entirely dependent on Danielle's personal approval. Claire was now a ghost in her own building. When Claire tried to schedule an all-hands meeting, only two people showed up. The others had been pulled into an 'urgent client emergency' orchestrated by Danielle. The betrayal was complete. The person Claire trusted most had turned her own creation against her.",
        "image_prompt": "A woman walking through a modern office, employees in the background whispering and looking away, blurred background, cold color palette.",
        "pexels_query": "woman walking office staff whispering",
        "music_mood": "isolated cold"
      },
      {
        "id": "s06_the_confrontation",
        "duration": 120,
        "narration": "Claire finally cornered Danielle in the boardroom. She demanded the books. Danielle didn't flinch. She sat back, crossed her legs, and slid a folder across the table. It wasn't the books. It was a formal buyout offer for Claire's remaining shares\u2014at ten cents on the dollar. Danielle calmly explained that the company was technically insolvent due to 'mismanagement' during Claire's absence. If Claire didn't sign, the company would go into forced liquidation, and Claire would be left with nothing but the debts. Danielle wasn't just stealing the company; she was forcing Claire to thank her for the exit. The mask had finally slipped. The 'best friend' was gone; only the predator remained.",
        "image_prompt": "Two women facing each other across a large mahogany boardroom table, intense eye contact, sharp lighting, cinematic composition.",
        "pexels_query": "two women arguing boardroom",
        "music_mood": "aggressive tension"
      },
      {
        "id": "s07_the_legal_battle",
        "duration": 120,
        "narration": "Claire refused to sign. She hired a forensic accountant and a high-stakes litigator. But Danielle was three steps ahead. Every piece of evidence Claire produced was countered by a document Claire had already signed. The 'Vanguard Consulting Group' was a shell company owned by Danielle's brother. The money was gone, laundered through a series of offshore accounts. The legal battle dragged on for months, draining what little savings Claire had left. Danielle used the company's own legal budget\u2014Claire's money\u2014to fight Claire. It was a war of attrition, and Danielle had the bigger arsenal. Claire realized that winning the legal battle might mean destroying the very company she loved.",
        "image_prompt": "Stack of legal folders and documents on a desk, a magnifying glass resting on them, dusty office environment, shafts of light.",
        "pexels_query": "legal documents magnifying glass",
        "music_mood": "slow heavy"
      },
      {
        "id": "s08_the_breaking_point",
        "duration": 90,
        "narration": "The breaking point came when Danielle filed for a restraining order, claiming Claire was 'harassing' the employees and creating a hostile work environment. Claire was barred from entering her own office. She stood on the sidewalk, watching the logo she had designed being scraped off the front door to be replaced by a new, corporate-cold rebrand: 'Sienna Global.' Danielle was erasing Claire from the history of her own brand. Grief, which had once blinded Claire, now fueled a cold, sharp clarity. She couldn't save Sienna Botanicals. But she could make sure Danielle didn't get to keep the crown.",
        "image_prompt": "A woman standing on a city sidewalk looking at a glass storefront where a logo is being removed, reflection of the city, sad but determined expression.",
        "pexels_query": "woman standing sidewalk looking storefront",
        "music_mood": "emotional shift"
      },
      {
        "id": "s09_the_counterstrike",
        "duration": 120,
        "narration": "Claire stopped fighting for the company and started fighting for the truth. She reached out to the one person Danielle hadn't been able to flip: the original chemist who had developed their core formulas. Together, they realized that Danielle had been cutting corners on ingredient quality to boost the margins for the buyout. The 'organic' skincare was no longer organic. Claire didn't go to the lawyers; she went to the regulatory boards and the press. She leaked the lab results showing the synthetic fillers Danielle had introduced. If Claire couldn't have her baby back, she would expose the monster it had become. The PR nightmare for 'Sienna Global' began within hours.",
        "image_prompt": "A laboratory setting with beakers and test tubes, a hand holding a digital tablet showing a graph with red lines, scientific lighting.",
        "pexels_query": "laboratory beakers research",
        "music_mood": "fast paced investigative"
      },
      {
        "id": "s10_the_collapse",
        "duration": 90,
        "narration": "The fallout was catastrophic. Major retailers pulled the products. Danielle's brother's 'consulting' firm was raided by the FBI following Claire's whistleblowing. The buyout Danielle had planned was now a liability. Investors fled. Danielle tried to reach out to Claire, offering a 'truce.' Claire didn't even answer the phone. Within six months, Sienna Global filed for bankruptcy. Danielle was left with a mountain of legal fees and a reputation that was radioactive in the industry. She had won the company, but she had inherited a burnt-out husk.",
        "image_prompt": "An empty warehouse with boxes stacked high, dust motes in the air, single light source, sense of abandonment.",
        "pexels_query": "empty warehouse boxes",
        "music_mood": "somber end"
      },
      {
        "id": "s11_the_new_beginning",
        "duration": 60,
        "narration": "Claire lost her company, but she kept her integrity. She started a new, smaller venture\u2014Claire\u2019s Essentials\u2014focusing on the community that had supported her from the beginning. She didn't look for a partner this time. She hired a team based on character, not just competence. She learned the hard way that trust is a privilege, not a default. The scars of Danielle's betrayal remained, but they were now the foundation of a much stronger, wiser woman.",
        "image_prompt": "Woman in a bright, small workshop mixing oils, smiling, sunlight streaming in, vibrant plants in the background.",
        "pexels_query": "woman workshop mixing oils",
        "music_mood": "uplifting acoustic"
      },
      {
        "id": "s12_the_final_lesson",
        "duration": 30,
        "narration": "If you're building something you love, remember: the person you trust most is the one who can hurt you most. Protect your vision, protect your finances, and never let grief sign a contract your future self will have to pay for. Because in business, your best friend might just be your biggest competitor in disguise.",
        "image_prompt": "Close-up of a woman's eyes, direct look at the camera, calm and powerful, soft focus background.",
        "pexels_query": "woman eyes close up powerful",
        "music_mood": "final powerful"
      }
    ]
  },
  "crimeledger": {
    "title": "The $50 Million Ponzi Scheme Nobody Saw Coming",
    "narration_wpm": 150,
    "voice": "serious_male",
    "scenes": [
      {
        "id": "s01_hook",
        "duration": 40,
        "narration": "He promised returns that beat the market every single month. He was the golden boy of the investment world, a man who transformed modest savings into generational wealth. But behind the luxury cars and the glass-fronted offices lay a void\u2014a $50 million hole that was swallowing people's lives whole. This is the autopsy of the Apex Fund, a Ponzi scheme so perfectly executed that even the experts didn't see the collapse coming until it was far too late.",
        "image_prompt": "Cinematic wide shot of a luxury glass office building at night, interior lights glowing, sharp architectural lines, dramatic low angle, cool blue tones.",
        "pexels_query": "luxury office building night glass",
        "music_mood": "dark cinematic"
      },
      {
        "id": "s02_the_visionary",
        "duration": 90,
        "narration": "Meet Julian Vance. In 2018, he was just another analyst at a mid-tier firm. But Julian had something others didn't: an irresistible charisma and a 'proprietary algorithm' he claimed could predict market volatility with 99% accuracy. He launched the Apex Fund in a small, prestigious office in Greenwich, Connecticut. He didn't advertise. He didn't cold call. He relied on the most powerful marketing tool in the world: exclusivity. To get into Apex, you had to be invited. And once you were in, you felt like you'd finally made it to the inner circle.",
        "image_prompt": "A charismatic man in a tailored suit standing in front of a wall of digital stock tickers, confident posture, warm but professional lighting.",
        "pexels_query": "businessman suit stock tickers",
        "music_mood": "confident steady"
      },
      {
        "id": "s03_the_pitch",
        "duration": 120,
        "narration": "The pitch was simple: 'Alpha without the Beta.' Julian claimed he wasn't just trading stocks; he was trading information. He used complex terminology that sounded profound but meant very little\u2014'asymmetric risk-adjusted arbitrage' and 'quantum-layer sentiment analysis.' To the wealthy retirees and small-business owners who flocked to him, it sounded like genius. He promised a steady 1.5% return every single month, regardless of whether the market was up or down. And for the first two years, he delivered exactly that. On paper, at least. Therefore, the word began to spread.",
        "image_prompt": "Close-up of a digital tablet showing a steady upward graph, 'APEX FUND' logo at the top, high-end desk environment, expensive fountain pen in frame.",
        "pexels_query": "investment graph tablet luxury desk",
        "music_mood": "pulsing momentum"
      },
      {
        "id": "s04_the_golden_years",
        "duration": 120,
        "narration": "By 2021, the Apex Fund was the talk of the town. Julian was appearing on local news as a 'financial prodigy.' He moved the headquarters to a sprawling estate and started hosting legendary 'Investor Appreciation' galas. People weren't just reinvesting their profits; they were liquidating their 401ks and selling their homes to give Julian more capital. He was the savior of the middle class, the man who had 'hacked' the system. But the truth was far simpler. There was no algorithm. There was no trading. There was only a very large spreadsheet and a constant influx of new money to pay out the old.",
        "image_prompt": "A lavish gala event, people in black tie attire holding champagne glasses, bokeh lights in background, sense of extreme wealth and celebration.",
        "pexels_query": "lavish gala party champagne",
        "music_mood": "grand orchestral"
      },
      {
        "id": "s05_the_red_flags",
        "duration": 100,
        "narration": "But every Ponzi scheme has its cracks. The first red flag came from a junior auditor who noticed that Apex\u2019s trades never seemed to appear on the public ledger in the volumes Julian claimed. When the auditor raised the issue, he was fired within the hour. Then there were the 'reporting delays.' Monthly statements that used to arrive on the 1st started slipping to the 5th, then the 10th. Julian blamed 'upgrades to the quantum server.' But the reality was that the inflow of new investors was starting to slow down, and the math was no longer working in his favor.",
        "image_prompt": "A hand holding a crumpled financial statement, red ink markings, shadow cast over the paper, dramatic side lighting.",
        "pexels_query": "crumpled paper financial statement red ink",
        "music_mood": "tense rhythmic"
      },
      {
        "id": "s06_the_whispers",
        "duration": 120,
        "narration": "Whispers began to circulate in private forums. One investor tried to withdraw $2 million for a real estate deal and was told it would take sixty days due to 'liquidity locks' in the new algorithm. Julian was personally calling worried clients, using his charm to convince them that the delay was actually a sign of the fund's strength. 'We're protecting your capital from a temporary market glitch,' he\u2019d say. It worked for some. But for others, the smell of smoke was becoming unmistakable. Therefore, a small group of investors decided to hire a private investigator.",
        "image_prompt": "Low-light shot of a man in a trench coat looking at a wall of photos and notes connected by red string, detective noir style.",
        "pexels_query": "detective wall photos red string",
        "music_mood": "noir mystery"
      },
      {
        "id": "s07_the_investigation",
        "duration": 120,
        "narration": "The investigator didn't find a sophisticated trading operation. He found a ghost town. The 'research department' Julian touted was just a room full of empty desks. The server room was barely a closet with a few consumer-grade routers. The investigator followed the money trail, which led not to Wall Street, but to a series of shell companies in the Cayman Islands and a private jet lease that was costing $200,000 a month. The 'proprietary algorithm' was Julian himself, manually typing numbers into a PDF template every month. The trap was set, but Julian was already planning his exit.",
        "image_prompt": "A darkened server room with blinking red lights, shadows stretching across the floor, cold and industrial feel.",
        "pexels_query": "dark server room red lights",
        "music_mood": "dark tech drone"
      },
      {
        "id": "s08_the_house_of_cards",
        "duration": 100,
        "narration": "In March 2024, the house of cards finally folded. A major institutional investor, suspicious of the delays, filed a formal complaint with the SEC. Within 48 hours, federal agents were at the doors of the Apex Fund. But Julian wasn't there. He had anticipated the move. While the agents were seizing empty hard drives and decorative furniture, Julian was on a private flight to a non-extradition country, carrying $5 million in untraceable cryptocurrency. He had left behind 400 devastated families and a $50 million crater in the local economy.",
        "image_prompt": "Federal agents in windbreakers with 'FBI' on the back entering a building, yellow crime scene tape in the foreground, chaotic scene.",
        "pexels_query": "fbi agents entering building crime scene",
        "music_mood": "intense percussive"
      },
      {
        "id": "s09_the_collapse",
        "duration": 80,
        "narration": "The news hit the community like a bomb. Retirees who had planned to spend their final years in comfort found themselves facing foreclosure. College funds evaporated. The 'Apex Family' Julian had cultivated was now a group of strangers united only by their ruin. The galas, the champagne, the 'exclusive' invitations\u2014it was all revealed for what it was: a high-priced illusion designed to lower their guard while he reached for their wallets.",
        "image_prompt": "An elderly couple sitting at a kitchen table, looking at documents with devastated expressions, single overhead light, somber and raw.",
        "pexels_query": "elderly couple sad kitchen table",
        "music_mood": "melancholy piano"
      },
      {
        "id": "s10_the_victims",
        "duration": 100,
        "narration": "We spoke to one victim, a schoolteacher who lost her entire life savings of $400,000. She told us, 'He didn't just take my money. He took my future. I trusted him because he went to my church. He looked me in the eye and told me my children would be taken care of.' Her story was mirrored by hundreds of others. Julian hadn't just stolen capital; he had weaponized social trust, using the community's own bonds to bind them to his lie. Therefore, the anger that followed was as much about the betrayal as the loss.",
        "image_prompt": "Close-up of a person's hands clenching a tissue, blurred background, focus on the tension in the hands.",
        "pexels_query": "hands clenching tissue crying",
        "music_mood": "emotional strings"
      },
      {
        "id": "s11_the_manhunt",
        "duration": 120,
        "narration": "The manhunt for Julian Vance lasted fourteen months. He was tracked through three different continents, moving between luxury rentals and obscure hostels. But Julian\u2019s greatest weakness was his own ego. He couldn't resist logging into his old trading forums to brag about his 'perfect exit.' The FBI's cyber-crime unit traced the IP to a coastal town in Montenegro. In a coordinated raid with local authorities, Julian was arrested while eating lunch at a waterfront caf\u00e9. He wasn't the golden boy anymore. He was a fugitive with nowhere left to run.",
        "image_prompt": "A man being led away in handcuffs by police on a sunny European street, ocean in the background, high contrast, paparazzi feel.",
        "pexels_query": "man arrested handcuffs police",
        "music_mood": "driving suspense"
      },
      {
        "id": "s12_the_trial",
        "duration": 80,
        "narration": "The trial of Julian Vance was a media circus. He attempted to argue that he was a victim of market forces, that his algorithm had simply 'malfunctioned.' But the evidence was overwhelming. The forensic accountants showed the clear path of the money\u2014from the bank accounts of teachers and nurses directly into Julian's luxury lifestyle. The jury took less than three hours to find him guilty on all 42 counts of wire fraud and money laundering. He was sentenced to 25 years in federal prison.",
        "image_prompt": "A courtroom scene, a judge's gavel hitting the sound block, motion blur, dramatic lighting from the side.",
        "pexels_query": "courtroom gavel judge",
        "music_mood": "heavy finality"
      },
      {
        "id": "s13_the_justice",
        "duration": 40,
        "narration": "But for the victims, justice was a cold comfort. Only $3 million of the $50 million was ever recovered. The rest had been spent or hidden too deeply to find. Julian Vance is currently serving his time, but the lives he destroyed will take generations to rebuild. The Apex Fund remains a haunting reminder that in the world of finance, if it sounds too good to be true, it\u2019s because it\u2019s a lie.",
        "image_prompt": "Close-up of a prison cell door closing, cold grey steel, bars in focus, dramatic shadows.",
        "pexels_query": "prison cell door bars",
        "music_mood": "somber drone"
      },
      {
        "id": "s14_final_warning",
        "duration": 30,
        "narration": "The next Julian Vance is already out there, refining his pitch. Don't be his next victim. Verify the trades, ignore the hype, and never invest money you can't afford to lose to a 'genius' with a secret algorithm. Because the only thing Julian Vance was a genius at was taking what wasn't his.",
        "image_prompt": "A silhouetted figure in a suit standing in a dark hallway, light coming from the end, mysterious and cautionary.",
        "pexels_query": "silhouette businessman dark hallway",
        "music_mood": "dark final"
      }
    ]
  },
  "mindtactics": {
    "title": "7 Signs Someone Is Manipulating You Right Now",
    "narration_wpm": 160,
    "voice": "calm_authoritative_female",
    "scenes": [
      {
        "id": "s01_hook",
        "duration": 30,
        "narration": "They aren't raising their voice. They aren't threatening you. In fact, they might even be smiling. But in the background, they are systematically dismantling your sense of reality. Manipulation is the art of getting someone to do what you want, while making them think it was their own idea. Today, we're revealing the seven psychological red flags that someone is pulling your strings\u2014and how to cut them before it's too late.",
        "image_prompt": "Cinematic close-up of a hand moving a wooden puppet's strings in a dimly lit theater, focus on the strings and the hand, moody atmosphere.",
        "pexels_query": "puppet strings hand manipulation",
        "music_mood": "calm mysterious"
      },
      {
        "id": "s02_invisible_strings",
        "duration": 60,
        "narration": "The most dangerous manipulators don't look like villains. They look like friends, partners, or mentors. They use your own empathy, your loyalty, and your desire for harmony against you. This isn't about healthy persuasion; it's about control. By the end of this video, you will have a psychological armor that makes you immune to these tactics. Let's start with the most intoxicating one: the Love Bomb.",
        "image_prompt": "A person looking at their own reflection in a cracked mirror, multiple facets of the same face, soft focus, introspective lighting.",
        "pexels_query": "person mirror reflection cracked",
        "music_mood": "ethereal steady"
      },
      {
        "id": "s03_sign_1_love_bombing",
        "duration": 110,
        "narration": "Sign number one: Love Bombing. At the start of a relationship, they shower you with excessive affection, praise, and attention. It feels like a fairy tale. But this isn't about love; it's about debt. They are creating a psychological 'surplus' so that when they later treat you poorly, you'll think, 'They were so good to me before, they must just be having a bad day.' It\u2019s a way of conditioning you to crave their approval. If it feels too fast and too intense, it probably is.",
        "image_prompt": "A person surrounded by a sea of red roses, but the thorns are visible and sharp, dramatic lighting, vibrant colors.",
        "pexels_query": "person roses thorns sharp",
        "music_mood": "warm but slightly off"
      },
      {
        "id": "s04_sign_2_gaslighting",
        "duration": 110,
        "narration": "Sign number two: Gaslighting. This is the calculated attempt to make you doubt your own memory, perception, or sanity. They\u2019ll say things like 'I never said that,' or 'You're being too sensitive.' When you have proof, they\u2019ll flip it on you, calling you 'paranoid.' Therefore, you start to rely on *their* version of events more than your own. The goal is to make you lose your psychological footing so you have to lean on them to stand up.",
        "image_prompt": "A person trying to catch fog with their hands, blurred environment, hazy and confusing atmosphere.",
        "pexels_query": "person catching fog hands confusion",
        "music_mood": "unsettling drone"
      },
      {
        "id": "s05_sign_3_guilt_tripping",
        "duration": 110,
        "narration": "Sign number three: The Weaponized Guilt Trip. They make their happiness your responsibility. If you don't do what they want, they act as the victim. 'After everything I've done for you, you can't do this one thing for me?' It turns every boundary you set into a personal attack against them. You end up saying 'yes' not because you want to, but because the weight of the guilt they've placed on you is too heavy to carry. It\u2019s a form of emotional blackmail.",
        "image_prompt": "A heavy iron chain wrapped around a heart-shaped object, dark background, metallic textures, cold lighting.",
        "pexels_query": "iron chain heart object",
        "music_mood": "heavy rhythmic"
      },
      {
        "id": "s06_sign_4_moving_goalposts",
        "duration": 110,
        "narration": "Sign number four: Moving the Goalposts. No matter how hard you try to please them, it\u2019s never enough. As soon as you meet one of their demands, they create a new, even more difficult one. They keep you in a state of constant striving. If you're always trying to prove your worth to someone, they have successfully manipulated you into a position of subservience. You are running a race where they control the finish line\u2014and they keep moving it further away.",
        "image_prompt": "A person running on a treadmill that is tilted upwards, sweat on their brow, minimalist gym setting, high contrast.",
        "pexels_query": "person running treadmill uphill",
        "music_mood": "tense momentum"
      },
      {
        "id": "s07_sign_5_isolation",
        "duration": 110,
        "narration": "Sign number five: Systematic Isolation. A manipulator's worst enemy is a strong support system. They will quietly start to drive wedges between you and your family or friends. They\u2019ll say things like, 'Your sister doesn't really understand our relationship,' or 'Your friends are just jealous of you.' They want you in a vacuum where they are the only voice you hear. Therefore, when they manipulate you, there\u2019s no one there to tell you that what\u2019s happening isn't normal.",
        "image_prompt": "A single bright lightbulb in a large, empty, dark room, casting long shadows, sense of solitude.",
        "pexels_query": "lightbulb empty dark room",
        "music_mood": "isolated quiet"
      },
      {
        "id": "s08_sign_6_silent_treatment",
        "duration": 110,
        "narration": "Sign number six: The Silent Treatment. This is a form of emotional withdrawal used as punishment. They stop talking to you, not because they need space, but to force you to apologize\u2014even when you\u2019ve done nothing wrong. It creates a deep anxiety and a fear of abandonment. You end up groveling just to get them to acknowledge your existence again. It\u2019s a power move designed to show you that your emotional well-being is entirely at their mercy.",
        "image_prompt": "A person sitting on a chair facing a blank wall, shadow of the chair stretching long, minimalist and stark.",
        "pexels_query": "person chair blank wall shadow",
        "music_mood": "low frequency drone"
      },
      {
        "id": "s09_sign_7_triangulation",
        "duration": 110,
        "narration": "Finally, sign number seven: Triangulation. They bring a third person into your dynamic to create competition and insecurity. They might constantly compare you to an ex, or a 'perfect' colleague. By making you feel replaceable, they ensure you work even harder to keep their favor. It\u2019s the ultimate divide-and-conquer strategy. They sit in the middle while you and the third party compete for their attention, giving them total control over the narrative.",
        "image_prompt": "A triangle formed by three people silhouetted against a bright background, one person at the top, two at the base, geometric composition.",
        "pexels_query": "three people silhouette triangle",
        "music_mood": "complex layered"
      },
      {
        "id": "s10_protection_exit",
        "duration": 60,
        "narration": "If you recognize these signs, the first step is to trust your intuition. Manipulators rely on you ignoring your 'gut feeling.' Start setting small boundaries and watch their reaction. A healthy person will respect a 'no.' A manipulator will escalate their tactics. Your peace of mind is not a bargaining chip. Knowledge is your first line of defense. Now that you can see the strings, you can choose to walk away. You are the only one who should have a hand in your own story.",
        "image_prompt": "Close-up of a hand cutting a thick rope with a sharp pair of scissors, bright sunlight, empowering feel.",
        "pexels_query": "hand cutting rope scissors",
        "music_mood": "empowering final"
      }
    ]
  }
}
print('Scenes per channel:')
for ch, s in SCRIPTS.items():
    print(f'  {ch}: {len(s["scenes"])} scenes')

In [ ]:
from chatterbox.tts import ChatterboxTTS

print('Loading Chatterbox TTS...')
tts = ChatterboxTTS.from_pretrained(device=device)
print('✅ TTS loaded')

VOICE_STYLES = {
    'twistedtruths': {'exaggeration': 0.7, 'cfg_weight': 3.0},
    'crimeledger':   {'exaggeration': 0.5, 'cfg_weight': 4.0},
    'mindtactics':   {'exaggeration': 0.4, 'cfg_weight': 3.5}
}

for channel, script in SCRIPTS.items():
    style = VOICE_STYLES[channel]
    print(f'\n🎙️ {channel} TTS...')
    for scene in tqdm(script['scenes'], desc=channel):
        out = BASE / channel / 'audio' / f"{scene['id']}.wav"
        if out.exists():
            continue
        try:
            wav = tts.generate(
                scene['narration'][:500],
                exaggeration=style['exaggeration'],
                cfg_weight=style['cfg_weight']
            )
            sf.write(str(out), wav.squeeze().cpu().numpy(), tts.sr)
        except Exception as e:
            print(f'  ERR {scene["id"]}: {e}')

    count = len(list((BASE/channel/'audio').glob('*.wav')))
    print(f'✅ {channel}: {count} audio files')
    
del tts
torch.cuda.empty_cache()
print('GPU cache cleared')

In [ ]:
from diffusers import FluxPipeline

print('Loading FLUX.1-schnell...')
pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.bfloat16
).to(device)
print('✅ FLUX loaded')

for channel, script in SCRIPTS.items():
    print(f'\n🖼️ {channel} images...')
    for scene in tqdm(script['scenes'], desc=channel):
        out = BASE / channel / 'images' / f"{scene['id']}.png"
        if out.exists():
            continue
        prompt = scene.get('image_prompt', scene['narration'][:200])
        full_prompt = (f"{prompt}, cinematic film still, "
            "dramatic chiaroscuro lighting, dark teal amber palette, "
            "film noir aesthetics, photorealistic, anamorphic bokeh, "
            "8K resolution, Arri Alexa, emotional tension")
        try:
            img = pipe(
                full_prompt,
                guidance_scale=0.0,
                num_inference_steps=4,
                width=1344, height=768
            ).images[0]
            img.save(str(out))
        except Exception as e:
            print(f'  ERR {scene["id"]}: {e}')
    
    count = len(list((BASE/channel/'images').glob('*.png')))
    print(f'✅ {channel}: {count} images')
    torch.cuda.empty_cache()

del pipe
torch.cuda.empty_cache()
print('Done!')

In [ ]:
import shutil, zipfile

print('📦 Zipping outputs...')
for channel in ['twistedtruths','crimeledger','mindtactics']:
    zip_path = BASE / f'{channel}_assets.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in (BASE/channel).rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(BASE/channel))
    size = zip_path.stat().st_size // 1024 // 1024
    print(f'✅ {channel}_assets.zip — {size}MB')

print('\n=== SUMMARY ===')
for channel in ['twistedtruths','crimeledger','mindtactics']:
    a = len(list((BASE/channel/'audio').glob('*.wav')))
    i = len(list((BASE/channel/'images').glob('*.png')))
    print(f'{channel}: {a} audio, {i} images')